In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry

import numpy as np
#import inspect     # - zum suchen von funktionencodes


#Kirchhoff-Love mit Hellan-Herrmann-Johnson-Method
#https://docu.ngsolve.org/ngs24/SaS/Kirchhoff_Love_plate.html


def kirchhoff_love(auflager_liste,l=1,b=1,radius=0.025,E=70e9,nu=0.22,t=0.003,rho=2500,step=40):
        # - l = 2 #m Länge
        # - b = 1 #m Breite

        #radius = 0.025 # Radius der Kreise für Auflager
    
    shape = MoveTo(0,0).Rectangle(l,b).Face()

    shape.edges.name="free"

    #circle = Circle((0,0),0.10).Face()
    #circle.edges.name="circles"

        # - auflager_liste = [(radius,radius),(l-radius,b-radius),(radius,b-radius),(l-radius,radius)]

    # auflager_liste = [(-a+radius,-b+radius),(-a/2,b/2),(-a/2,-b/2),(a/2,-b/2)]
    #auflager_liste = [(-0.3,-0.1),(-0.1,0)]

    rect = shape 
    for mittelpunkt in auflager_liste:
        circle = Circle(mittelpunkt,radius).Face()
        circle.edges.name = "dirichlet"
        rect = rect - circle

    geo = OCCGeometry(rect,dim=2)

    #Draw(rect)
    mesh = Mesh(geo.GenerateMesh(maxh=l/40))
    mesh.Curve(3)
    # Draw(mesh)


        # - E  = 70e9      #Glas ~ 70 GPa Elastizitätsmodul
        # - nu = 0.22
        # - t  = 0.003     #Dicke [m]
        # - rho = 2500   #kg/m^3 Dichte einer Aluminiumplatte
    g = 9.81

    q = rho * t * g     #Eigengewicht der Platte - rechte Seite der PDE

    Db = E*t**3/(12*(1-nu**2))

    def D(A):
        return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

    def Dinv(A):
        return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))

    #Funktionenraum/raeume
    order = 3

    V = HDivDiv(mesh, order=order-1,dirichlet="dirichlet")
    Q = H1(mesh, order=order, dirichlet="dirichlet")
    X = V * Q

    (sigma,w),(tau,v)= X.TnT()

    n = specialcf.normal(2)     #normalvektor

    def tang(u):
        return u - (u*n)*n


    #schwache Form
    a = BilinearForm(X,symmetric=True)
    a += InnerProduct(Dinv(sigma),tau) * dx
    a += div(sigma)*Grad(v) * dx
    a += div(tau) * Grad(w) * dx
    a += - (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
    a.Assemble()

    L = LinearForm(X)
    L += q * v *dx
    L.Assemble()

    #Lösen
    gf_solution = GridFunction(X)
    gf_solution.vec.data = a.mat.Inverse(X.FreeDofs(),inverse="") * L.vec

    #Visualisieren
    gf_sigma, gf_w = gf_solution.components
    #Draw(gf_sigma, mesh,name="sigma")
    
    #HAUPT-VISUALISIERUNG
    # Draw(gf_w, mesh, name="disp",deformation=True, euler_angles=[-60,5,30])


    #zum Weiterrechnen wollen wir auf die errechneten Daten zugreifen können


        # - step = 40           #TODO warum ab 41 out of domain?

    #erzeugt gitter für gewünschte werte
    x_points = np.linspace(0,l, step)
    y_points = np.linspace(0,b, step)
    X, Y = np.meshgrid(x_points, y_points)


    #speichert (x,y,z) in result_matrix für alle (x,y) in X,Y
    result_matrix = np.zeros((step, step),dtype=object)
    for i in range(step):
        for j in range(step):
            result_matrix[i, j] = (float(X[i,j]), float(Y[i,j]), gf_w(X[i, j], Y[i, j]))

    # print(result_matrix)

    return result_matrix


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

BaseWebGuiScene

[[(0.0, 0.0, -3.3194945791217933e-10)
  (0.05128205128205128, 0.0, -4.8897869335956936e-05)
  (0.10256410256410256, 0.0, -0.000709512758515984)
  (0.15384615384615385, 0.0, -0.0017310264235433702)
  (0.20512820512820512, 0.0, -0.002970778413686403)
  (0.2564102564102564, 0.0, -0.004360944827279117)
  (0.3076923076923077, 0.0, -0.005850349137659408)
  (0.358974358974359, 0.0, -0.007396166962886577)
  (0.41025641025641024, 0.0, -0.008960952624214553)
  (0.4615384615384615, 0.0, -0.010511239591268873)
  (0.5128205128205128, 0.0, -0.012016824871966135)
  (0.5641025641025641, 0.0, -0.0134504020095135)
  (0.6153846153846154, 0.0, -0.014787385025255362)
  (0.6666666666666666, 0.0, -0.016005839498995315)
  (0.717948717948718, 0.0, -0.017086472894412043)
  (0.7692307692307692, 0.0, -0.018012655645760742)
  (0.8205128205128205, 0.0, -0.01877045577406326)
  (0.8717948717948718, 0.0, -0.019348676608422183)
  (0.923076923076923, 0.0, -0.019738891392580126)
  (0.9743589743589743, 0.0, -0.01993547116